Let's start with using the open sourced OPENAI CLIP model

In [9]:
import re
import torch
import torch.nn.functional as F
import torchvision.transforms as T
from torchvision.transforms import Compose, Resize, ToTensor, Normalize, InterpolationMode
import clip
from PIL import Image
from captum.attr import visualization
from CLIP.clip.simple_tokenizer import SimpleTokenizer as _Tokenizer
from transformers import CLIPTokenizer
_tokenizer = _Tokenizer()
tokenizer = CLIPTokenizer.from_pretrained("openai/clip-vit-base-patch32")
from clip import tokenize
from open_clip import tokenize
import cv2
import pickle
import base64
from io import BytesIO
device = "cuda" if torch.cuda.is_available() else "cpu"
clipmodel, preprocess = clip.load("ViT-B/16", device=device)

In [10]:
def pre_caption(caption, max_words=50):
    caption = re.sub(
        r"([.!\"()*#:;~])",       
        ' ',
        caption.lower(),
    )
    caption = re.sub(
        r"\s{2,}",
        ' ',
        caption,
    )
    caption = caption.rstrip('\n') 
    caption = caption.strip(' ')

    #truncate caption
    caption_words = caption.split(' ')
    if len(caption_words)>max_words:
        caption = ' '.join(caption_words[:max_words])
            
    return caption

In [11]:
def attention_layer(q, k, v, num_heads=1, attn_mask=None):
    "Compute 'Scaled Dot Product Attention'"
    tgt_len, bsz, embed_dim = q.shape
    head_dim = embed_dim // num_heads
    scaling = float(head_dim) ** -0.5
    q = q * scaling
    
    q = q.contiguous().view(tgt_len, bsz * num_heads, head_dim).transpose(0, 1)
    k = k.contiguous().view(-1, bsz * num_heads, head_dim).transpose(0, 1)
    v = v.contiguous().view(-1, bsz * num_heads, head_dim).transpose(0, 1)
    attn_output_weights = torch.bmm(q, k.transpose(1, 2))
    if attn_mask is not None:
        attn_output_weights += attn_mask
    attn_output_weights = F.softmax(attn_output_weights, dim=-1)
    attn_output = torch.bmm(attn_output_weights, v)
    assert list(attn_output.size()) == [bsz * num_heads, tgt_len, head_dim]
    attn_output = attn_output.transpose(0, 1).contiguous().view(tgt_len, bsz, embed_dim)
    attn_output_weights = attn_output_weights.view(bsz, num_heads, tgt_len, -1)
    attn_output_weights = attn_output_weights.sum(dim=1) / num_heads
    return attn_output, attn_output_weights
    
def clip_encode_text_dense(text, n):
    x = clipmodel.token_embedding(text).type(clipmodel.dtype)  # [batch_size, n_ctx, d_model]
    attn_mask=clipmodel.build_attention_mask().to(dtype=x.dtype, device=x.device)
    x = x + clipmodel.positional_embedding.type(clipmodel.dtype)
    x = x.permute(1, 0, 2)  # NLD -> LND
    x = torch.nn.Sequential(*clipmodel.transformer.resblocks[:-n])(x)

    #####################
    attns = []
    atten_outs = []
    vs = []
    qs = []
    ks = []
    for TR in clipmodel.transformer.resblocks[-n:]:
        x_in = x
        x = TR.ln_1(x_in)
        linear = torch._C._nn.linear    
        q, k, v = linear(x, TR.attn.in_proj_weight, TR.attn.in_proj_bias).chunk(3, dim=-1)
        attn_output, attn = attention_layer(q, k, v, 1, attn_mask=attn_mask) # num_head=1
        attns.append(attn)
        atten_outs.append(attn_output)
        vs.append(v)
        qs.append(q)
        ks.append(k)
        
        x_after_attn = linear(attn_output, TR.attn.out_proj.weight, TR.attn.out_proj.bias)       
        x = x_after_attn + x_in
        x = x + TR.mlp(TR.ln_2(x))
            
    x = x.permute(1, 0, 2)  # LND -> NLD
    x = clipmodel.ln_final(x).type(clipmodel.dtype)

    # x.shape = [batch_size, n_ctx, transformer.width]
    # take features from the eot embedding (eot_token is the highest number in each sequence)
    x = x[torch.arange(x.shape[0]), text.argmax(dim=-1)] @ clipmodel.text_projection
    return x, (qs, ks, vs), attns, atten_outs

In [ ]:

def sim_qk(q, k):
    q_cls = F.normalize(q[eos_position,0,:], dim=-1) 
    k_patch = F.normalize(k[:,0,:], dim=-1)

    cosine_qk = (q_cls * k_patch).sum(-1)  
    cosine_qk = (cosine_qk-cosine_qk.min()) / (cosine_qk.max()-cosine_qk.min())
    return cosine_qk

def grad_eclip(c, qs, ks, vs, attn_outputs, eos_position):
    ## gradient on last attention output
    tmp_maps = []
    for q, k, v, attn_output in zip(qs, ks, vs, attn_outputs):
        grad = torch.autograd.grad(
            c,
            attn_output,
            retain_graph=True)[0]
        grad_cls = grad[eos_position,0,:]
        # just use the gradient on the cls token position  
        cosine_qk = sim_qk(q, k)
        # print("[cosine_qk]:", cosine_qk.shape) # 77
        tmp_maps.append((grad_cls * v[:,0,:] * cosine_qk[:,None]).sum(-1))

    emap = (F.relu_(torch.stack(tmp_maps, dim=0).sum(0)))
    emap = emap[1:eos_position].flatten()
    emap = emap / emap.sum()
    return emap

def grad_eclip_per_dimension(text_emb_30d, qs, ks, vs, atten_outs, eos_position):
    all_grad_emaps = []
    for dim_idx in range(30): 
        cosine_dim = text_emb_30d[0, dim_idx]
        tmp_maps = []
        for q, k, v, attn_output in zip(qs, ks, vs, atten_outs):
            grad = torch.autograd.grad(
                cosine_dim,
                attn_output,
                retain_graph=True)[0]
            grad_cls = grad[eos_position, 0, :]
            q_cls = F.normalize(q[eos_position, 0, :], dim=-1) 
            k_patch = F.normalize(k[:, 0, :], dim=-1)
            cosine_qk = (q_cls * k_patch).sum(-1)  
            cosine_qk = (cosine_qk - cosine_qk.min()) / (cosine_qk.max() - cosine_qk.min())
            
            tmp_maps.append((grad_cls * v[:, 0, :] * cosine_qk[:, None]).sum(-1))

        emap = (F.relu_(torch.stack(tmp_maps, dim=0).sum(0)))
        emap = emap[1:eos_position].flatten()

        if emap.sum() > 0:
            emap = emap / emap.sum()
        else:
            emap = torch.zeros_like(emap)
            
        all_grad_emaps.append(emap)
    
    return all_grad_emaps

def self_attn(attns, eos_position):
    ## attn map of cls token on each word
    # attn: 1,77,77
    
    attn = attns[-1][0, eos_position,:].flatten()
    attn = attn / attn.sum()
    return attn

def load_linear_model(model_path):
    with open(model_path, "rb") as f:
        lin_model = pickle.load(f)
    W = torch.tensor(lin_model.W, dtype=torch.float32, device=device)  # (512, 30)
    b = torch.tensor(lin_model.b, dtype=torch.float32, device=device)  # (1, 30)
    W.requires_grad_(False)
    b.requires_grad_(False)
    return W, b

def extract_frame(video_path, frame_index):
    cap = cv2.VideoCapture(video_path)
    for i in range(frame_index + 1):
        success, frame = cap.read()
        if not success:
            cap.release()
    cap.release()
    frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    img = Image.fromarray(frame_rgb).convert("RGB")

    return img

def load_folder_mapping(folder_list_path):
    folder_mapping = {}
    with open(folder_list_path, 'r') as file:
        for line in file:
            number, folder_name = line.strip().split()
            folder_mapping[int(number)] = folder_name
    return folder_mapping

def read_txt_file(file_path):
    with open(file_path, 'r', encoding='utf-8') as file:
        return file.read()
    
def pil_to_base64(img):
    buffered = BytesIO()
    img.save(buffered, format="PNG")
    img_base64 = base64.b64encode(buffered.getvalue()).decode("utf-8")
    return img_base64

def image_to_base64(image_path):
    with open(image_path, "rb") as f:
        img_data = f.read()
        img_base64 = base64.b64encode(img_data).decode("utf-8")
    return img_base64

In [ ]:
import os
import matplotlib.pyplot as plt
from PIL import Image
import numpy as np


Y = np.loadtxt("data/LLM_deepseek/deepseek_spose_embedding_sorted_final.txt")
top3_indices = np.argsort(Y, axis=1)[:, -3:]

print("Top 3 indices for each video:", top3_indices[:10])
video_model_path = "data/grad_cam/fastl2lir_LLM_deepseek_video.pkl"
text_model_path = "data/grad_cam/fastl2lir_LLM_deepseek_text.pkl"
W_video, b_video = load_linear_model(video_model_path)
W_text, b_text = load_linear_model(text_model_path)
folder_list_path = "data/folder_list/folder_list.txt"
transciptions_folder = 'data/transcriptions_selected'
folder_mapping = load_folder_mapping(folder_list_path)
category_names = [
    '手部做家务', '运动', '面部 综合', '说话讨论', '手部动作', 
    '多人交互', '全身（运动）', '粉碎 危险动作', '驾驶', '手操纵科技',
    '看护婴儿', '水相关', '艺术表演', '积极欢呼', '写 绘画',
    '球类', '护理', '孩童游乐', '户外', '声音',
    '厨房', '暴力', '全身（静止）', '面部', '工具操作',
    '交易', '吃喝', '宗教', '休息', '打扫'
]




Top 3 indices for each video: [[19  3 12]
 [ 2 19  3]
 [19  3 12]
 [ 2 19  3]
 [15  0 21]
 [ 4  3 13]
 [ 3  1  6]
 [ 5  7 21]
 [ 2  8  6]
 [ 5  4  3]]


In [ ]:
video_index = 229
dim_idx = 0

video_path = os.path.join(f"data/videos_selected", folder_mapping.get(int(video_index)))
for file_name in os.listdir(video_path):
    video = os.path.join(video_path, file_name)
img = extract_frame(video, frame_index=0)
category = folder_mapping.get(video_index)
path = os.path.join(transciptions_folder, category)
for file_name in os.listdir(path):
    text_path = os.path.join(path, file_name)
    sentence = read_txt_file(text_path)
img_preprocessed = preprocess(img).cuda().unsqueeze(0)
img_embedding = clipmodel.encode_image(img_preprocessed)
img_embedding = F.normalize(img_embedding, dim=-1)
img_embedding_30d = img_embedding.float() @ W_video + b_video
text_processed = clip.tokenize([sentence])
text_tokens=_tokenizer.encode(sentence)
text_tokens_decoded=[_tokenizer.decode([a]) for a in text_tokens]
print("Text Tokens:", text_tokens)
print(text_tokens_decoded)
token_ids = tokenize(sentence)[0].tolist()
decoded_tokens = [tokenizer.decode([tid]) for tid in token_ids if tid != 0]  # 忽略padding
print("Token IDs:", token_ids)
print("Decoded Tokens:", decoded_tokens)
x, (qs, ks, vs), attns, atten_outs = clip_encode_text_dense(text_processed.cuda(), n=8)
text_embedding = F.normalize(x, dim=-1)
text_embedding_30d = text_embedding.float() @ W_text + b_text
eos_position = text_processed.argmax(dim=-1) 
grad_emaps_30d = grad_eclip_per_dimension(text_embedding_30d, qs, ks, vs, atten_outs, eos_position)
# grad_emaps_30d = grad_eclip(text_embedding_30d, qs, ks, vs, atten_outs, eos_position)

img_base64 = pil_to_base64(img)
image_html = f'''
<h2>Visualize</h2>
<img src="data:image/png;base64,{img_base64}" alt="Overview Image" style="width: 400px; height:auto;">
<hr>
'''
all_html_blocks = [image_html]
gradcam_folder = f"data/grad_cam/gradCAM_results_LLM_deepseek/video_{video_index}"

actual_dim_idx = dim_idx
grad_emap = grad_emaps_30d[actual_dim_idx]
category_name = category_names[actual_dim_idx]

gradcam_image_path = None
if os.path.exists(gradcam_folder):
    for filename in os.listdir(gradcam_folder):
        if f"dim_{actual_dim_idx}" in filename or f"dimension_{actual_dim_idx}" in filename:
            gradcam_image_path = os.path.join(gradcam_folder, filename)
            break

vis_data_records = [visualization.VisualizationDataRecord(
    grad_emap, 0, 0, 0, 0, 0, text_tokens_decoded, 1
)]

html_object = visualization.visualize_text(vis_data_records)
html_block = f'''
    <div>
        {html_object.data}
    </div>
    '''
with open(f"data/grad_cam/results_LLM_deepseek/heatmap_{video_index}_{actual_dim_idx + 1}.html", "w", encoding="utf-8") as f:
    f.write(html_block)


Text Tokens: [8305, 29018, 320, 48321, 5168, 633, 518, 23243]
['somebody ', 'removes ', 'a ', 'fondue ', 'pot ', 'from ', 'the ', 'burner ']
Token IDs: [49406, 8305, 29018, 320, 48321, 5168, 633, 518, 23243, 49407, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
Decoded Tokens: ['<|startoftext|>', 'somebody', 'removes', 'a', 'fondue', 'pot', 'from', 'the', 'burner', '<|endoftext|>']


True Label,Predicted Label,Attribution Label,Attribution Score,Word Importance
0,0 (0.00),0,0.00,somebody removes a fondue pot from the burner


In [ ]:
for video_index in range(355):
    target_dim = top3_indices[video_index]
    video_path = os.path.join(f"data/videos_selected", folder_mapping.get(int(video_index)))

    for file_name in os.listdir(video_path):
        video = os.path.join(video_path, file_name)
    img = extract_frame(video, frame_index=0)

    category = folder_mapping.get(video_index)
    path = os.path.join(transciptions_folder, category)
    for file_name in os.listdir(path):
        text_path = os.path.join(path, file_name)
        sentence = read_txt_file(text_path)

    img_preprocessed = preprocess(img).cuda().unsqueeze(0)
    img_embedding = clipmodel.encode_image(img_preprocessed)
    img_embedding = F.normalize(img_embedding, dim=-1)
    img_embedding_30d = img_embedding.float() @ W_video + b_video
    text_processed = clip.tokenize([sentence])


    text_tokens=_tokenizer.encode(sentence)
    text_tokens_decoded=[_tokenizer.decode([a]) for a in text_tokens]
    print("Text Tokens:", text_tokens)
    print(text_tokens_decoded)

    token_ids = tokenize(sentence)[0].tolist()

    decoded_tokens = [tokenizer.decode([tid]) for tid in token_ids if tid != 0]  # 忽略padding
    print("Token IDs:", token_ids)
    print("Decoded Tokens:", decoded_tokens)


    x, (qs, ks, vs), attns, atten_outs = clip_encode_text_dense(text_processed.cuda(), n=8)
    text_embedding = F.normalize(x, dim=-1)
    text_embedding_30d = text_embedding.float() @ W_text + b_text
    eos_position = text_processed.argmax(dim=-1) 

    grad_emaps_30d = grad_eclip_per_dimension(text_embedding_30d, qs, ks, vs, atten_outs, eos_position)
    # grad_emaps_30d = grad_eclip(text_embedding_30d, qs, ks, vs, atten_outs, eos_position)
    print(f"Target dimensions: {target_dim}")
    print(f"Category: {[category_names[dim] for dim in target_dim]}")
    
    img_base64 = pil_to_base64(img)
    image_html = f'''
    <h2>Visualize</h2>
    <img src="data:image/png;base64,{img_base64}" alt="Overview Image" style="width: 400px; height:auto;">
    <hr>
    '''

    all_html_blocks = [image_html]
    gradcam_folder = f"data/grad_cam/gradCAM_results_LLM_deepseek/video_{video_index}"
    
    for i, dim_idx in enumerate(target_dim):
        actual_dim_idx = dim_idx
        grad_emap = grad_emaps_30d[actual_dim_idx]
        category_name = category_names[actual_dim_idx]
        
        gradcam_image_path = None
        if os.path.exists(gradcam_folder):
            for filename in os.listdir(gradcam_folder):
                if f"dim_{actual_dim_idx}" in filename or f"dimension_{actual_dim_idx}" in filename:
                    gradcam_image_path = os.path.join(gradcam_folder, filename)
                    break
        
        vis_data_records = [visualization.VisualizationDataRecord(
            grad_emap, 0, 0, 0, 0, 0, text_tokens_decoded, 1
        )]
        
        html_object = visualization.visualize_text(vis_data_records)
        html_block = f'''
            <div>
                {html_object.data}
            </div>
            <hr>
            '''
        with open(f"data/grad_cam/results_LLM_deepseek/heatmap_{video_index}_{actual_dim_idx + 1}.html", "w", encoding="utf-8") as f:
            f.write(html_block)
